## Model Initialize

In [65]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatGoogleGenerativeAI(model = "gemini-3.5-flash-lite",
                                max_tokens = 512,
                                api_key = os.getenv("GOOGLE_API_KEY"))

## Output Parsing

### Using For Control The Response Type, Example str -> json

In [66]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [67]:
from langchain_core.prompts import ChatPromptTemplate

In [68]:
prompt_template = ChatPromptTemplate.from_template(review_template)

In [69]:
prompt_template

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})])

In [70]:
prompt = prompt_template.format_messages(text = customer_review)
response = model.invoke(prompt).content

In [71]:
print(response[0]['text'])

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ]
}
```


In [72]:
# This Is String Response Not Json
type(response[0]['text'])

str

In [73]:
from pydantic import BaseModel, Field

In [74]:
class Response(BaseModel):
    gift: bool = Field(description="Was the item purchased\
                                as a gift for someone else? \
                                Answer True if yes,\
                                False if not or unknown.")
    
    delivery_days: int = Field(description="How many days\
                                    did it take for the product\
                                    to arrive? If this \
                                    information is not found,\
                                    output -1.")
    
    price_value: str = Field(description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

In [75]:
structured_model = model.with_structured_output(Response)

In [76]:
prompt

[HumanMessage(content="For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.\n\n", additiona

In [77]:
response = structured_model.invoke(prompt)

In [78]:
response

Response(gift=True, delivery_days=2, price_value="['It\\'s slightly more expensive than the other leaf blowers out there, but I think it\\'s worth it for the extra features.']")

In [81]:
print(response.gift)
print(response.delivery_days)
print(response.price_value)

True
2
['It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features.']
